<a href="https://colab.research.google.com/github/sinamahdavi/aml-2025-mistake-detection/blob/sanam/notebooks/egovlp_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%cd /content
!rm -rf code

!git clone --recursive -b sanam https://github.com/sinamahdavi/aml-2025-mistake-detection.git code
%cd code


/content
Cloning into 'code'...
remote: Enumerating objects: 727, done.
remote: Counting objects: 100% (105/105), done.
remote: Compressing objects: 100% (102/102), done.
remote: Total 727 (delta 8), reused 57 (delta 3), pack-reused 622 (from 1)
Receiving objects: 100% (727/727), 4.84 MiB | 15.74 MiB/s, done.
Resolving deltas: 100% (428/428), done.
Submodule 'annotations' (https://github.com/CaptainCook4D/annotations) registered for path 'annotations'
Cloning into '/content/code/annotations'...
remote: Enumerating objects: 152, done.        
remote: Counting objects: 100% (152/152), done.        
remote: Compressing objects: 100% (98/98), done.        
remote: Total 152 (delta 75), reused 108 (delta 46), pack-reused 0 (from 0)        
Receiving objects: 100% (152/152), 793.14 KiB | 15.25 MiB/s, done.
Resolving deltas: 100% (75/75), done.
Submodule path 'annotations': checked out '0e9a108be2cbcbcbd592e7418c0ab9c16232d27a'
/content/code


In [2]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [3]:
# Setup EgoVLP features from Google Drive
import os
import shutil

os.chdir('/content/code')
!mkdir -p data/video/egovlp checkpoints

# IMPORTANT: First, add the shared folder to your Drive:
# 1. Open: https://drive.google.com/drive/folders/1Wb7fgwS94VZ27weF4UGQDy9GaTQWHW_C
# 2. Right-click the folder → "Add shortcut to Drive" → Choose "My Drive"
# 3. Then update the path below to where you added it (e.g., /content/drive/MyDrive/egovlp)

EGOVLP_DRIVE_PATH = "/content/drive/MyDrive/egovlp"  # Update if you added it to a different location
EGOVLP_LOCAL_PATH = "data/video/egovlp"

print("🔗 Setting up EgoVLP features from Google Drive...")

# Check if path exists
if not os.path.exists(EGOVLP_DRIVE_PATH):
    print(f"❌ Path not found: {EGOVLP_DRIVE_PATH}")
    print("\n💡 To fix:")
    print("   1. Open the shared folder: https://drive.google.com/drive/folders/1Wb7fgwS94VZ27weF4UGQDy9GaTQWHW_C")
    print("   2. Right-click → 'Add shortcut to Drive' → Choose 'My Drive'")
    print("   3. Update EGOVLP_DRIVE_PATH above with the actual path")
    print("\n   Or search for it:")
    !find /content/drive/MyDrive -name "*egovlp*" -type d 2>/dev/null | head -5
else:
    # Remove existing local path if needed
    if os.path.exists(EGOVLP_LOCAL_PATH) and not os.path.islink(EGOVLP_LOCAL_PATH):
        shutil.rmtree(EGOVLP_LOCAL_PATH) if os.path.isdir(EGOVLP_LOCAL_PATH) else os.remove(EGOVLP_LOCAL_PATH)

    # Create symlink
    if not os.path.exists(EGOVLP_LOCAL_PATH):
        os.symlink(EGOVLP_DRIVE_PATH, EGOVLP_LOCAL_PATH)

    # Verify
    files = [f for f in os.listdir(EGOVLP_LOCAL_PATH) if f.endswith('.npz')]
    print(f"✅ Found {len(files)} EgoVLP feature files")
    print("✅ EgoVLP features ready!")

🔗 Setting up EgoVLP features from Google Drive...
✅ Found 384 EgoVLP feature files
✅ EgoVLP features ready!


In [4]:
# Install required packages
!pip install torcheval tabulate
!pip install loguru

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.2/179.2 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 4.7 MB/s eta 0:00:00


## Train Models with EgoVLP Backbone

Training three model variants (MLP, Transformer, LSTM) on both splits (recordings and step).


### Train MLP Models


In [5]:
# Train MLP + EgoVLP on recordings split
import os
os.chdir('/content/code')
!python train_er.py --variant MLP --backbone egovlp --split recordings --batch_size 8 --num_epochs 10 --lr 1e-3 --weight_decay 1e-3


-------------------------------------------------------------
Training step model and testing on step level
Train args: {'num_workers': 8, 'pin_memory': False, 'shuffle': True, 'batch_size': 8}
Test args: {'num_workers': 8, 'pin_memory': False, 'shuffle': False, 'batch_size': 1}
{'batch_size': 8, 'test_batch_size': 1, 'num_epochs': 10, 'lr': 0.001, 'weight_decay': 0.001, 'ckpt': None, 'seed': 42, 'backbone': 'egovlp', 'ckpt_directory': './checkpoints', 'split': 'recordings', 'variant': 'MLP', 'model_name': None, 'task_name': 'error_recognition', 'error_category': None, 'modality': ['video'], 'device': None}
-------------------------------------------------------------
Loaded annotations...... 
Loading recording ids from recordings_combined_splits.json
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this D

In [6]:
# Train MLP + EgoVLP on step split
import os
os.chdir('/content/code')
!python train_er.py --variant MLP --backbone egovlp --split step --batch_size 8 --num_epochs 10 --lr 1e-3 --weight_decay 1e-3

-------------------------------------------------------------
Training step model and testing on step level
Train args: {'num_workers': 8, 'pin_memory': False, 'shuffle': True, 'batch_size': 8}
Test args: {'num_workers': 8, 'pin_memory': False, 'shuffle': False, 'batch_size': 1}
{'batch_size': 8, 'test_batch_size': 1, 'num_epochs': 10, 'lr': 0.001, 'weight_decay': 0.001, 'ckpt': None, 'seed': 42, 'backbone': 'egovlp', 'ckpt_directory': './checkpoints', 'split': 'step', 'variant': 'MLP', 'model_name': None, 'task_name': 'error_recognition', 'error_category': None, 'modality': ['video'], 'device': None}
-------------------------------------------------------------
Loaded annotations...... 
Loading recording ids from recordings_combined_splits.json
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoa

### Train Transformer Models


In [7]:
# Train Transformer + EgoVLP on recordings split
import os
os.chdir('/content/code')
!python train_er.py --variant Transformer --backbone egovlp --split recordings --batch_size 8 --num_epochs 10 --lr 1e-3 --weight_decay 1e-3


-------------------------------------------------------------
Training step model and testing on step level
Train args: {'num_workers': 8, 'pin_memory': False, 'shuffle': True, 'batch_size': 8}
Test args: {'num_workers': 8, 'pin_memory': False, 'shuffle': False, 'batch_size': 1}
{'batch_size': 8, 'test_batch_size': 1, 'num_epochs': 10, 'lr': 0.001, 'weight_decay': 0.001, 'ckpt': None, 'seed': 42, 'backbone': 'egovlp', 'ckpt_directory': './checkpoints', 'split': 'recordings', 'variant': 'Transformer', 'model_name': None, 'task_name': 'error_recognition', 'error_category': None, 'modality': ['video'], 'device': None}
-------------------------------------------------------------
Loaded annotations...... 
Loading recording ids from recordings_combined_splits.json
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than wha

In [8]:
# Train Transformer + EgoVLP on step split
import os
os.chdir('/content/code')
!python train_er.py --variant Transformer --backbone egovlp --split step --batch_size 8 --num_epochs 10 --lr 1e-3 --weight_decay 1e-3

-------------------------------------------------------------
Training step model and testing on step level
Train args: {'num_workers': 8, 'pin_memory': False, 'shuffle': True, 'batch_size': 8}
Test args: {'num_workers': 8, 'pin_memory': False, 'shuffle': False, 'batch_size': 1}
{'batch_size': 8, 'test_batch_size': 1, 'num_epochs': 10, 'lr': 0.001, 'weight_decay': 0.001, 'ckpt': None, 'seed': 42, 'backbone': 'egovlp', 'ckpt_directory': './checkpoints', 'split': 'step', 'variant': 'Transformer', 'model_name': None, 'task_name': 'error_recognition', 'error_category': None, 'modality': ['video'], 'device': None}
-------------------------------------------------------------
Loaded annotations...... 
Loading recording ids from recordings_combined_splits.json
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this

### Train LSTM Models


In [9]:
# Train LSTM + EgoVLP on recordings split
import os
os.chdir('/content/code')
!python train_lstm.py --variant LSTM --backbone egovlp --split recordings --batch_size 8 --num_epochs 10 --lr 1e-3 --weight_decay 1e-3


Training LSTM model for Error Recognition (Step 2b)
Backbone: egovlp
Split: recordings
Learning Rate: 0.001
Epochs: 10
Device: cuda
Loaded annotations...... 
Loading recording ids from recordings_combined_splits.json
Loaded annotations...... 
Loading recording ids from recordings_combined_splits.json
Loaded annotations...... 
Loading recording ids from recordings_combined_splits.json
Train Epoch: 1, Progress: 496/497, Loss: 1.225275: 100% 497/497 [02:08<00:00,  3.87it/s]
val Progress: 681/86: 100% 86/86 [00:19<00:00,  4.37it/s]
----------------------------------------------------------------
val Sub Step Level Metrics: {'precision': 0.34965034965034963, 'recall': 0.2127659574468085, 'f1': 0.26455026455026454, 'accuracy': 0.591776798825257, 'auc': np.float64(0.5009922717297968), 'pr_auc': tensor(0.3461)}
val Step Level Metrics: {'precision': 0.10810810810810811, 'recall': 0.5714285714285714, 'f1': 0.18181818181818182, 'accuracy': 0.5813953488372093, 'auc': np.float64(0.5045207956600362)

In [10]:
# Train LSTM + EgoVLP on step split
import os
os.chdir('/content/code')
!python train_lstm.py --variant LSTM --backbone egovlp --split step --batch_size 8 --num_epochs 10 --lr 1e-3 --weight_decay 1e-3


Training LSTM model for Error Recognition (Step 2b)
Backbone: egovlp
Split: step
Learning Rate: 0.001
Epochs: 10
Device: cuda
Loaded annotations...... 
Loading recording ids from recordings_combined_splits.json
Loaded annotations...... 
Loading recording ids from recordings_combined_splits.json
Loaded annotations...... 
Loading recording ids from recordings_combined_splits.json
Train Epoch: 1, Progress: 468/469, Loss: 1.203320: 100% 469/469 [02:04<00:00,  3.75it/s]
val Progress: 774/97: 100% 97/97 [00:22<00:00,  4.41it/s]
----------------------------------------------------------------
val Sub Step Level Metrics: {'precision': 0.31776913099870296, 'recall': 0.9959349593495935, 'f1': 0.48180924287118976, 'accuracy': 0.31912144702842377, 'auc': np.float64(0.5125916173934466), 'pr_auc': tensor(0.3178)}
val Step Level Metrics: {'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'accuracy': 0.6597938144329897, 'auc': 0.0, 'pr_auc': tensor(-0.)}
-----------------------------------------------------

## Comprehensive Analysis

After training, run this section to generate all analysis visualizations and tables.


In [11]:
# Setup and Imports
import os
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from torch.utils.data import DataLoader
import warnings
warnings.filterwarnings('ignore')

os.chdir('/content/code')

# Import project modules
from base import fetch_model, test_er_model
from constants import Constants as const
from dataloader.CaptainCookStepDataset import CaptainCookStepDataset, collate_fn, step_sequence_collate_fn

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✅ Imports successful!")


✅ Imports successful!


In [12]:
# Configuration - EgoVLP only
MODELS = [const.MLP_VARIANT, const.TRANSFORMER_VARIANT, const.LSTM_VARIANT]
BACKBONE = const.EGOVLP  # Only EgoVLP
SPLITS = [const.RECORDINGS_SPLIT, const.STEP_SPLIT]
THRESHOLDS = {const.RECORDINGS_SPLIT: 0.4, const.STEP_SPLIT: 0.6}
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Create results directory
os.makedirs("results/egovlp_analysis", exist_ok=True)
os.makedirs("results/egovlp_analysis/tables", exist_ok=True)
os.makedirs("results/egovlp_analysis/charts", exist_ok=True)
os.makedirs("results/egovlp_analysis/heatmaps", exist_ok=True)

print(f"Device: {DEVICE}")
print(f"Models: {MODELS}")
print(f"Backbone: {BACKBONE}")
print(f"Splits: {SPLITS}")
print(f"Thresholds: {THRESHOLDS}")


Device: cuda
Models: ['MLP', 'Transformer', 'LSTM']
Backbone: egovlp
Splits: ['recordings', 'step']
Thresholds: {'recordings': 0.4, 'step': 0.6}


### Helper Functions


In [13]:
class EvalConfig:
    """Simple config class for evaluation."""
    def __init__(self, backbone="egovlp", variant="MLP", split="recordings", device="cuda"):
        self.backbone = backbone
        self.modality = const.VIDEO
        self.phase = "test"
        self.segment_length = 1
        self.segment_features_directory = "data/"
        self.ckpt_directory = ""
        self.split = split
        self.batch_size = 1
        self.test_batch_size = 1
        self.seed = 1000
        self.device = device
        self.variant = variant
        self.task_name = const.ERROR_RECOGNITION


def find_best_checkpoint(variant, backbone, split="recordings"):
    """Find the best checkpoint for a given variant and backbone."""
    official_patterns = [
        f"checkpoints/error_recognition_best/{variant}/{backbone}/*{split}*.pt",
        f"checkpoints/error_recognition_best/{variant}/{backbone}/*.pt",
    ]
    trained_patterns = [
        f"checkpoints/error_recognition/{variant}/{backbone}/*_best.pt",
        f"checkpoints/error_recognition/{variant}/{backbone}/*best*.pt",
        f"checkpoints/error_recognition/{variant}/{backbone}/*{split}*.pt",
        f"checkpoints/error_recognition/{variant}/{backbone}/*.pt"
    ]

    for pattern in official_patterns + trained_patterns:
        ckpts = glob.glob(pattern)
        if ckpts:
            best_ckpts = [c for c in ckpts if '_best' in c.lower() or 'best' in c.lower()]
            if best_ckpts:
                return sorted(best_ckpts, key=os.path.getmtime)[-1]
            return sorted(ckpts, key=os.path.getmtime)[-1]
    return None


def evaluate_model(variant, backbone, split, device="cuda", threshold=0.4):
    """Evaluate a model and return metrics."""
    ckpt_path = find_best_checkpoint(variant, backbone, split)
    if not ckpt_path or not os.path.exists(ckpt_path):
        print(f"⚠️  No checkpoint found for {variant} + {backbone} + {split}")
        return None

    print(f"📊 Evaluating {variant} + {backbone} on {split} split...")

    config = EvalConfig(backbone=backbone, variant=variant, split=split, device=device)
    model = fetch_model(config)
    model.load_state_dict(torch.load(ckpt_path, map_location=device))
    model.eval()

    test_dataset = CaptainCookStepDataset(config, const.TEST, split)
    if variant in [const.LSTM_VARIANT, const.GRU_VARIANT]:
        test_loader = DataLoader(test_dataset, batch_size=1, collate_fn=step_sequence_collate_fn)
    else:
        test_loader = DataLoader(test_dataset, batch_size=1, collate_fn=collate_fn)

    criterion = torch.nn.BCEWithLogitsLoss()
    test_losses, sub_step_metrics, step_metrics = test_er_model(
        model, test_loader, criterion, device,
        phase="test",
        step_normalization=True,
        sub_step_normalization=True,
        threshold=threshold
    )

    return {
        'variant': variant,
        'backbone': backbone,
        'split': split,
        'accuracy': step_metrics[const.ACCURACY] * 100,
        'precision': step_metrics[const.PRECISION] * 100,
        'recall': step_metrics[const.RECALL] * 100,
        'f1': step_metrics[const.F1] * 100,
        'auc': step_metrics[const.AUC] * 100
    }

print("✅ Helper functions defined!")


✅ Helper functions defined!


In [14]:
# Evaluate all combinations
all_results = []

for variant in MODELS:
    for split in SPLITS:
        threshold = THRESHOLDS[split]
        result = evaluate_model(variant, BACKBONE, split, DEVICE, threshold)
        if result:
            all_results.append(result)

# Convert to DataFrame
df_all = pd.DataFrame(all_results)

# Save raw results
df_all.to_csv("results/egovlp_analysis/all_results.csv", index=False)
print(f"\n✅ Evaluated {len(all_results)} model configurations")
print(f"Results saved to: results/egovlp_analysis/all_results.csv")


📊 Evaluating MLP + egovlp on recordings split...
Loaded annotations...... 
Loading recording ids from recordings_combined_splits.json


test Progress: 38340/671: 100%|██████████| 671/671 [00:21<00:00, 30.62it/s]


----------------------------------------------------------------
test Sub Step Level Metrics: {'precision': 0.3701098034058781, 'recall': 0.8929959546283811, 'f1': 0.5233236490412551, 'accuracy': 0.465075639019301, 'auc': np.float64(0.6626031891756882), 'pr_auc': tensor(0.3657)}
test Step Level Metrics: {'precision': 0.3828996282527881, 'recall': 0.8547717842323651, 'f1': 0.5288831835686778, 'accuracy': 0.45305514157973176, 'auc': np.float64(0.5269854289298466), 'pr_auc': tensor(0.3795)}
----------------------------------------------------------------
📊 Evaluating MLP + egovlp on step split...
Loaded annotations...... 
Loading recording ids from recordings_combined_splits.json


test Progress: 42347/798: 100%|██████████| 798/798 [00:24<00:00, 32.42it/s]


----------------------------------------------------------------
test Sub Step Level Metrics: {'precision': 0.3174881387688495, 'recall': 0.8862831111860975, 'f1': 0.46750472800089, 'accuracy': 0.434835997827473, 'auc': np.float64(0.6464387623064738), 'pr_auc': tensor(0.3132)}
test Step Level Metrics: {'precision': 0.36363636363636365, 'recall': 0.4979919678714859, 'f1': 0.42033898305084744, 'accuracy': 0.5714285714285714, 'auc': np.float64(0.5843336917798699), 'pr_auc': tensor(0.3377)}
----------------------------------------------------------------
📊 Evaluating Transformer + egovlp on recordings split...
Loaded annotations...... 
Loading recording ids from recordings_combined_splits.json


...

----------------------------------------------------------------
test Sub Step Level Metrics: {'precision': 0.2799253784211396, 'recall': 1.0, 'f1': 0.4374089038947621, 'accuracy': 0.2799253784211396, 'auc': np.float64(0.491442009960261), 'pr_auc': tensor(0.2799)}
test Step Level Metrics: {'precision': 0.31958762886597936, 'recall': 0.4979919678714859, 'f1': 0.3893249607535322, 'accuracy': 0.5125313283208021, 'auc': np.float64(0.5056583346134996), 'pr_auc': tensor(0.3158)}
----------------------------------------------------------------
📊 Evaluating LSTM + egovlp on recordings split...
Loaded annotations...... 
Loading recording ids from recordings_combined_splits.json


test Progress: 671/671: 100%|██████████| 671/671 [00:23<00:00, 28.57it/s]


----------------------------------------------------------------
test Sub Step Level Metrics: {'precision': 0.3591654247391952, 'recall': 1.0, 'f1': 0.5285087719298246, 'accuracy': 0.3591654247391952, 'auc': np.float64(0.5), 'pr_auc': tensor(0.3592)}
test Step Level Metrics: {'precision': 0.3591654247391952, 'recall': 1.0, 'f1': 0.5285087719298246, 'accuracy': 0.3591654247391952, 'auc': np.float64(0.5), 'pr_auc': tensor(0.3592)}
----------------------------------------------------------------
📊 Evaluating LSTM + egovlp on step split...
Loaded annotations...... 
Loading recording ids from recordings_combined_splits.json


test Progress: 798/798: 100%|██████████| 798/798 [00:26<00:00, 30.26it/s]


----------------------------------------------------------------
test Sub Step Level Metrics: {'precision': 0.31203007518796994, 'recall': 1.0, 'f1': 0.47564469914040114, 'accuracy': 0.31203007518796994, 'auc': np.float64(0.5), 'pr_auc': tensor(0.3120)}
test Step Level Metrics: {'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'accuracy': 0.6879699248120301, 'auc': np.float64(0.5), 'pr_auc': tensor(0.3120)}
----------------------------------------------------------------

✅ Evaluated 6 model configurations
Results saved to: results/egovlp_analysis/all_results.csv


### 1. Tables

In [15]:
def create_table_image(df, title, filename):
    """Create a PNG image of a table."""
    fig, ax = plt.subplots(figsize=(14, max(6, len(df) * 0.5 + 2)))
    ax.axis('tight')
    ax.axis('off')

    table_data = []
    for _, row in df.iterrows():
        table_data.append([
            row['variant'], row['split'],
            f"{row['accuracy']:.2f}", f"{row['precision']:.2f}",
            f"{row['recall']:.2f}", f"{row['f1']:.2f}", f"{row['auc']:.2f}"
        ])

    headers = ["Model", "Split", "Accuracy", "Precision", "Recall", "F1", "AUC"]
    table = ax.table(cellText=table_data, colLabels=headers, cellLoc='center', loc='center')
    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1.2, 2)

    for i in range(len(headers)):
        table[(0, i)].set_facecolor('#4CAF50')
        table[(0, i)].set_text_props(weight='bold', color='white')

    for i in range(1, len(table_data) + 1):
        for j in range(len(headers)):
            if i % 2 == 0:
                table[(i, j)].set_facecolor('#f0f0f0')

    plt.title(title, fontsize=16, fontweight='bold', pad=20)
    plt.savefig(filename, dpi=300, bbox_inches='tight', facecolor='white')
    plt.close()
    print(f"✅ Saved: {filename}")


# Create overall table
df_all_sorted = df_all.sort_values(['split', 'variant'])
csv_path = "results/egovlp_analysis/tables/comparison_all.csv"
df_all_sorted.to_csv(csv_path, index=False)
print(f"✅ Saved CSV: {csv_path}")

png_path = "results/egovlp_analysis/tables/comparison_all.png"
create_table_image(df_all_sorted, "EgoVLP Model Comparison - All Splits", png_path)

# Create tables for each split
for split in SPLITS:
    df_split = df_all[df_all['split'] == split].copy()
    df_split = df_split.sort_values(['variant'])

    csv_path = f"results/egovlp_analysis/tables/comparison_{split}.csv"
    df_split.to_csv(csv_path, index=False)
    print(f"✅ Saved CSV: {csv_path}")

    png_path = f"results/egovlp_analysis/tables/comparison_{split}.png"
    create_table_image(df_split, f"EgoVLP Model Comparison - {split.upper()} Split", png_path)

print("\n✅ All comparison tables created!")


✅ Saved CSV: results/egovlp_analysis/tables/comparison_all.csv
✅ Saved: results/egovlp_analysis/tables/comparison_all.png
✅ Saved CSV: results/egovlp_analysis/tables/comparison_recordings.csv
✅ Saved: results/egovlp_analysis/tables/comparison_recordings.png
✅ Saved CSV: results/egovlp_analysis/tables/comparison_step.csv
✅ Saved: results/egovlp_analysis/tables/comparison_step.png

✅ All comparison tables created!


### 2. Metrics Heatmaps


In [16]:
def create_heatmap(df, split, save_path):
    """Create metrics heatmap."""
    if split is None:
        df_split = df.copy()
        title = "EgoVLP Metrics Heatmap - All Splits"
    else:
        df_split = df[df['split'] == split].copy()
        title = f"EgoVLP Metrics Heatmap - {split.upper()} Split"

    metrics = ['accuracy', 'precision', 'recall', 'f1', 'auc']
    heatmap_data = []

    for variant in df_split['variant'].unique():
        if split is None:
            for s in df_split['split'].unique():
                subset = df_split[(df_split['variant'] == variant) & (df_split['split'] == s)]
                if len(subset) > 0:
                    row_data = [subset[m].values[0] for m in metrics]
                    heatmap_data.append({
                        'Model': f"{variant}_{s}",
                        **{m: row_data[i] for i, m in enumerate(metrics)}
                    })
        else:
            subset = df_split[df_split['variant'] == variant]
            if len(subset) > 0:
                row_data = [subset[m].values[0] for m in metrics]
                heatmap_data.append({
                    'Model': variant,
                    **{m: row_data[i] for i, m in enumerate(metrics)}
                })

    if not heatmap_data:
        print(f"⚠️  No data for heatmap: {save_path}")
        return

    df_heatmap = pd.DataFrame(heatmap_data)
    df_heatmap = df_heatmap.set_index('Model')

    plt.figure(figsize=(10, max(6, len(df_heatmap) * 0.6)))
    sns.heatmap(df_heatmap, annot=True, fmt='.2f', cmap='YlOrRd',
                cbar_kws={'label': 'Score (%)'}, linewidths=0.5, linecolor='gray')
    plt.title(title, fontsize=16, fontweight='bold', pad=20)
    plt.xlabel('Metrics', fontsize=12, fontweight='bold')
    plt.ylabel('Model', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"✅ Saved: {save_path}")


for split in SPLITS:
    save_path = f"results/egovlp_analysis/heatmaps/heatmap_{split}.png"
    create_heatmap(df_all, split, save_path)

save_path = "results/egovlp_analysis/heatmaps/heatmap_all.png"
create_heatmap(df_all, None, save_path)

print("\n✅ All heatmaps created!")


✅ Saved: results/egovlp_analysis/heatmaps/heatmap_recordings.png
✅ Saved: results/egovlp_analysis/heatmaps/heatmap_step.png
✅ Saved: results/egovlp_analysis/heatmaps/heatmap_all.png

✅ All heatmaps created!


### 3. Charts


In [17]:
def create_metrics_bar_chart(df, split, save_path):
    """Create bar chart comparing all metrics across models."""
    df_split = df[df['split'] == split].copy()

    if df_split.empty:
        print(f"⚠️  No data for split: {split}")
        return

    models = df_split['variant'].tolist()
    metrics = ['accuracy', 'precision', 'recall', 'f1', 'auc']

    fig, ax = plt.subplots(figsize=(12, 7))

    x = np.arange(len(models))
    width = 0.15
    colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4', '#FFEAA7']

    for i, metric in enumerate(metrics):
        values = df_split[metric].tolist()
        bars = ax.bar(x + i * width, values, width, label=metric.capitalize(),
                     color=colors[i], alpha=0.8)
        # Add value labels
        for bar in bars:
            height = bar.get_height()
            if height > 0:
                ax.text(bar.get_x() + bar.get_width()/2., height,
                       f'{height:.1f}',
                       ha='center', va='bottom', fontsize=9, fontweight='bold')

    ax.set_xlabel('Model', fontsize=12, fontweight='bold')
    ax.set_ylabel('Score (%)', fontsize=12, fontweight='bold')
    ax.set_title(f'EgoVLP Metrics Comparison - {split.upper()} Split', fontsize=14, fontweight='bold')
    ax.set_xticks(x + width * 2)
    ax.set_xticklabels(models, fontsize=11)
    ax.legend(loc='upper left', fontsize=10)
    ax.grid(axis='y', alpha=0.3)
    ax.set_ylim([0, 100])

    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"✅ Saved: {save_path}")


def create_individual_metric_charts(df, split, save_dir):
    """Create individual bar charts for each metric."""
    df_split = df[df['split'] == split].copy()

    if df_split.empty:
        print(f"⚠️  No data for split: {split}")
        return

    models = df_split['variant'].tolist()
    metrics = ['accuracy', 'precision', 'recall', 'f1', 'auc']
    colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4', '#FFEAA7']

    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    axes = axes.flatten()

    for idx, metric in enumerate(metrics):
        ax = axes[idx]
        values = df_split[metric].tolist()

        bars = ax.bar(models, values, color=colors[idx], alpha=0.8, edgecolor='black', linewidth=1.5)

        # Add value labels
        for bar in bars:
            height = bar.get_height()
            if height > 0:
                ax.text(bar.get_x() + bar.get_width()/2., height,
                       f'{height:.2f}%',
                       ha='center', va='bottom', fontsize=11, fontweight='bold')

        ax.set_ylabel('Score (%)', fontsize=11, fontweight='bold')
        ax.set_title(f'{metric.capitalize()}', fontsize=12, fontweight='bold')
        ax.set_xticklabels(models, fontsize=10)
        ax.grid(axis='y', alpha=0.3)
        ax.set_ylim([0, 100])

    # Remove empty subplot
    fig.delaxes(axes[5])

    plt.suptitle(f'EgoVLP Individual Metrics - {split.upper()} Split',
                 fontsize=16, fontweight='bold', y=0.995)
    plt.tight_layout()

    save_path = os.path.join(save_dir, f"individual_metrics_{split}.png")
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"✅ Saved: {save_path}")


# Create charts for each split
for split in SPLITS:
    # Combined metrics bar chart
    save_path = f"results/egovlp_analysis/charts/metrics_comparison_{split}.png"
    create_metrics_bar_chart(df_all, split, save_path)

    # Individual metric charts
    create_individual_metric_charts(df_all, split, "results/egovlp_analysis/charts")

print("\n✅ All metrics comparison charts created!")

✅ Saved: results/egovlp_analysis/charts/metrics_comparison_recordings.png
✅ Saved: results/egovlp_analysis/charts/individual_metrics_recordings.png
✅ Saved: results/egovlp_analysis/charts/metrics_comparison_step.png
✅ Saved: results/egovlp_analysis/charts/individual_metrics_step.png

✅ All metrics comparison charts created!


In [18]:
# Run error type analysis for each model
print("Running error type analysis...\n")

for variant in MODELS:
    for split in SPLITS:
        threshold = THRESHOLDS[split]
        ckpt_path = find_best_checkpoint(variant, BACKBONE, split)

        if ckpt_path and os.path.exists(ckpt_path):
            print(f"Analyzing {variant} + {BACKBONE} on {split} split...")
            cmd = f'python -m core.evaluate_error_types --variant {variant} --backbone {BACKBONE} --split {split} --ckpt "{ckpt_path}" --threshold {threshold} --save_csv'
            os.system(cmd)
            print()

print("✅ Error type analysis complete!")
print("Results saved to: results/error_type_analysis/")

# Create PNG images from CSV files
print("\nGenerating PNG images for error type analysis...")

def create_error_type_table_image(csv_path, png_path):
    """Create a PNG image from error type analysis CSV."""
    try:
        # Read CSV file
        with open(csv_path, 'r') as f:
            lines = f.readlines()

        # Find the "Per Error Type Metrics" section
        per_error_start = None
        for i, line in enumerate(lines):
            if "Per Error Type Metrics" in line:
                per_error_start = i + 1  # Skip header line
                break

        if per_error_start is None:
            print(f"⚠️  Could not find 'Per Error Type Metrics' section in {csv_path}")
            return

        # Parse the per error type metrics
        table_data = []
        for line in lines[per_error_start:]:
            line = line.strip()
            if not line:
                continue
            parts = line.split(',')
            if len(parts) >= 7:
                error_type = parts[0]
                count = parts[1]
                accuracy = parts[2]
                precision = parts[3]
                recall = parts[4]
                f1 = parts[5]
                auc = parts[6]
                table_data.append([error_type, count, accuracy, precision, recall, f1, auc])

        if not table_data:
            print(f"⚠️  No error type data found in {csv_path}")
            return

        # Create figure
        fig, ax = plt.subplots(figsize=(14, max(6, len(table_data) * 0.5 + 2)))
        ax.axis('tight')
        ax.axis('off')

        headers = ["Error Type", "Count", "Accuracy", "Precision", "Recall", "F1", "AUC"]
        table = ax.table(cellText=table_data, colLabels=headers, cellLoc='center', loc='center')
        table.auto_set_font_size(False)
        table.set_fontsize(10)
        table.scale(1.2, 2)

        # Style header
        for i in range(len(headers)):
            table[(0, i)].set_facecolor('#4CAF50')
            table[(0, i)].set_text_props(weight='bold', color='white')

        # Style rows (alternating colors)
        for i in range(1, len(table_data) + 1):
            for j in range(len(headers)):
                if i % 2 == 0:
                    table[(i, j)].set_facecolor('#f0f0f0')

        # Get variant, backbone, split from filename
        filename = os.path.basename(csv_path)
        parts = filename.replace('_error_type_analysis.csv', '').split('_')
        if len(parts) >= 3:
            variant_name = parts[0]
            backbone_name = parts[1]
            split_name = parts[2]
            title = f"Error Type Analysis - {variant_name} + {backbone_name} ({split_name})"
        else:
            title = "Error Type Analysis"

        plt.title(title, fontsize=16, fontweight='bold', pad=20)
        plt.savefig(png_path, dpi=300, bbox_inches='tight', facecolor='white')
        plt.close()
        print(f"✅ Saved PNG: {png_path}")
    except Exception as e:
        print(f"⚠️  Error creating PNG for {csv_path}: {e}")

# Create PNG directory
os.makedirs("results/error_type_analysis/png", exist_ok=True)

# Generate PNG for each CSV file
for variant in MODELS:
    for split in SPLITS:
        csv_filename = f"{variant}_{BACKBONE}_{split}_error_type_analysis.csv"
        csv_path = f"results/error_type_analysis/{csv_filename}"
        png_path = f"results/error_type_analysis/png/{csv_filename.replace('.csv', '.png')}"

        if os.path.exists(csv_path):
            create_error_type_table_image(csv_path, png_path)
        else:
            print(f"⚠️  CSV file not found: {csv_path}")

print("\n✅ All error type analysis PNG images created!")
print("PNG files saved to: results/error_type_analysis/png/")


Running error type analysis...

Analyzing MLP + egovlp on recordings split...

Analyzing MLP + egovlp on step split...

Analyzing Transformer + egovlp on recordings split...

Analyzing Transformer + egovlp on step split...

Analyzing LSTM + egovlp on recordings split...

Analyzing LSTM + egovlp on step split...

✅ Error type analysis complete!
Results saved to: results/error_type_analysis/

Generating PNG images for error type analysis...
✅ Saved PNG: results/error_type_analysis/png/MLP_egovlp_recordings_error_type_analysis.png
✅ Saved PNG: results/error_type_analysis/png/MLP_egovlp_step_error_type_analysis.png
✅ Saved PNG: results/error_type_analysis/png/Transformer_egovlp_recordings_error_type_analysis.png
✅ Saved PNG: results/error_type_analysis/png/Transformer_egovlp_step_error_type_analysis.png
✅ Saved PNG: results/error_type_analysis/png/LSTM_egovlp_recordings_error_type_analysis.png
✅ Saved PNG: results/error_type_analysis/png/LSTM_egovlp_step_error_type_analysis.png

✅ All erro

In [20]:
!zip -r results.zip /content/code/results

  adding: content/code/results/ (stored 0%)
  adding: content/code/results/backbone_comparison_recordings_charts.png (deflated 26%)
  adding: content/code/results/comprehensive_analysis/ (stored 0%)
  adding: content/code/results/comprehensive_analysis/heatmaps/ (stored 0%)
  adding: content/code/results/comprehensive_analysis/heatmaps/heatmap_recordings.png (deflated 14%)
  adding: content/code/results/comprehensive_analysis/heatmaps/heatmap_all.png (deflated 10%)
  adding: content/code/results/comprehensive_analysis/heatmaps/heatmap_step.png (deflated 14%)
  adding: content/code/results/comprehensive_analysis/tables/ (stored 0%)
  adding: content/code/results/comprehensive_analysis/tables/comparison_recordings.csv (deflated 45%)
  adding: content/code/results/comprehensive_analysis/tables/comparison_all.png (deflated 22%)
  adding: content/code/results/comprehensive_analysis/tables/comparison_recordings.png (deflated 24%)
  adding: content/code/results/comprehensive_analysis/tables/c